In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = logistic_model.predict(X_test)

y_probability = logistic_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
from src.evaluate import evaluate_model

logistic_results = evaluate_model(
    logistic_model,
    X_test,
    y_test
)

print(logistic_results)

Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=6,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBClassifier(
                n_estimators=400,
                learning_rate=0.05,
                max_depth=5,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=42
            )
        )
    ]
)

LightGBM

In [ ]:
from lightgbm import LGBMClassifier

lightgbm_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LGBMClassifier(
                n_estimators=400,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=31,
                random_state=42,
                verbosity=-1
            )
        )
    ]
)

Train All Models

In [ ]:
models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "XGBoost": xgb_model,
    "LightGBM": lightgbm_model
}

results = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(
        X_train,
        y_train
    )

    results[name] = evaluate_model(
        model,
        X_test,
        y_test
    )

Compare Models

In [ ]:
results_df = pd.DataFrame(results).T

results_df = results_df.sort_values(
    "ROC-AUC",
    ascending=False
)

results_df

Visualize:

In [ ]:
results_df[
    ["Precision", "Recall", "F1", "ROC-AUC"]
].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=30)
plt.tight_layout()

plt.show()

Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_model = models[
    results_df.index[0]
]

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test
)

plt.title(
    f"Confusion Matrix - {results_df.index[0]}"
)

plt.show()

ROC Curve

In [ ]:
from sklearn.metrics import RocCurveDisplay

plt.figure(figsize=(8, 6))

for name, model in models.items():

    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=name
    )

plt.title("ROC Curves")
plt.show()

Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [
        200,
        400,
        600
    ],

    "model__max_depth": [
        3,
        5,
        7
    ],

    "model__learning_rate": [
        0.01,
        0.05,
        0.1
    ]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    grid_search.best_params_
)

print(
    grid_search.best_score_
)

Save Model

In [ ]:
import joblib

final_model = grid_search.best_estimator_

joblib.dump(
    final_model,
    "models/churn_model.joblib"
)

In [ ]:
metadata = {
    "model": "XGBoost",
    "target": "Churn",
    "random_state": 42
}

joblib.dump(
    metadata,
    "models/model_metadata.joblib"
)

In [ ]:
import shap

model = final_model.named_steps["model"]
processor = final_model.named_steps["preprocessor"]

X_test_processed = processor.transform(X_test)

explainer = shap.TreeExplainer(model)

shap_values = explainer.shap_values(
    X_test_processed
)

In [ ]:
feature_names = (
    processor
    .get_feature_names_out()
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_processed,
    feature_names=feature_names
)

In [ ]:
customer = X_test.iloc[[0]]

prediction = final_model.predict(
    customer
)[0]

probability = final_model.predict_proba(
    customer
)[0, 1]

print("Prediction:", prediction)
print("Probability:", probability)